# Week 1: First LLM Lab — Web Page Summarizer

This notebook applies the Week 1 concepts: load credentials, fetch web content, and call an LLM to summarize it. It uses the shared scraper in `src/week1/` and the prompt template in `prompts/`.

**Prerequisites:** [Week 1 guide](docs/weeks/Week1.md), `.env` with `OPENAI_API_KEY`, and `uv sync` from repo root.

**Run order:** Execute cells from top to bottom.

## 1. Setup and imports

Add the repo root to the path so we can import the Week 1 scraper and the scripts helper.

In [ ]:
import os
import sys
from pathlib import Path

# Repo root: one level up from notebooks/, or cwd if already at root
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from dotenv import load_dotenv
load_dotenv(repo_root / ".env")

from scripts.api_client import get_openai_client
from week1.scraper import fetch_website_contents

print("Repo root:", repo_root)
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))

## 2. Load the summarization prompt

We use the template in `prompts/web-summarize.prompt.txt` and substitute the `content` variable.

In [ ]:
prompt_path = repo_root / "prompts" / "web-summarize.prompt.txt"
prompt_template = prompt_path.read_text(encoding="utf-8")

def build_summarize_prompt(content: str) -> str:
    return prompt_template.replace("{{content}}", content)

## 3. Fetch a page and summarize with the LLM

Fetch text from a URL (using the shared scraper), then send it to the model with the summarization prompt.

In [ ]:
url = "https://httpbin.org/html"  # or try any public article URL
raw_text = fetch_website_contents(url)
print(f"Fetched {len(raw_text)} characters from {url}")
print("First 200 chars:", raw_text[:200], "...")

messages = [
    {"role": "user", "content": build_summarize_prompt(raw_text)}
]
client = get_openai_client()
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=0.2,
    max_tokens=300,
)
summary = response.choices[0].message.content
print("\n--- Summary ---")
print(summary)

## 4. Optional: try another URL

Change `url` and re-run the previous cell to summarize a different page. Keep in mind the scraper truncates to 2000 characters.